
# GP Geo CV Result Analysis

Summarize restartable GP tuning results written by `GP Geo CV Tuning.ipynb`.


In [ ]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CWD = Path.cwd()
if CWD.name == "gridsearch":
    NOTEBOOK_DIR = CWD
elif (CWD / "gridsearch").exists() and CWD.name == "code":
    NOTEBOOK_DIR = CWD / "gridsearch"
elif (CWD / "code" / "gridsearch").exists():
    NOTEBOOK_DIR = CWD / "code" / "gridsearch"
else:
    NOTEBOOK_DIR = CWD
CODE_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(CODE_DIR))

from eval_workflow import completed_gp_results, summarize_cv_results


In [ ]:

GRID_PATH = CODE_DIR / "gridsearch" / "gp_geo_stage1_grid.csv"
STAGE1_RESULT_DIR = CODE_DIR / "eval_results" / "gp_stage1"
STAGE2_RESULT_DIR = CODE_DIR / "eval_results" / "gp_stage2"
FINAL_GP_PATH = CODE_DIR / "eval_results" / "gp_final_holdout.csv"
FINAL_KNN_PATH = CODE_DIR / "eval_results" / "knn_final_holdout.csv"

grid = pd.read_csv(GRID_PATH)

def stage_summary(result_dir, expected_folds):
    results = completed_gp_results(result_dir)
    if results.empty:
        return results, pd.DataFrame()
    summary = summarize_cv_results(results).merge(grid, on="param_id", how="left")
    summary["complete"] = summary["n_folds"] == expected_folds
    return results, summary.sort_values(["complete", "mean_mse"], ascending=[False, True])

stage1_results, stage1_summary = stage_summary(STAGE1_RESULT_DIR, expected_folds=3)
stage2_results, stage2_summary = stage_summary(STAGE2_RESULT_DIR, expected_folds=5)

len(stage1_results), len(stage2_results)


In [ ]:

stage1_summary.head(20)


In [ ]:

stage2_summary.head(20)


In [ ]:

def usable_summary():
    complete_stage2 = stage2_summary[stage2_summary["complete"]] if not stage2_summary.empty else pd.DataFrame()
    if not complete_stage2.empty:
        return "stage2", complete_stage2
    complete_stage1 = stage1_summary[stage1_summary["complete"]] if not stage1_summary.empty else pd.DataFrame()
    return "stage1", complete_stage1

active_stage, active_summary = usable_summary()
print(f"Using {active_stage} complete results for family bests")
active_summary.head(10)


In [ ]:

def best_rows_by_family(summary):
    if summary.empty:
        return pd.DataFrame()

    selectors = {
        "overall": np.ones(len(summary), dtype=bool),
        "no_time": summary["os_t"] == 0,
        "additive_ap": summary["ap_form"] == "add",
        "multiplicative_ap": summary["ap_form"] == "mult",
        "no_ap": summary["ap_form"] == "none",
    }
    rows = []
    for label, mask in selectors.items():
        subset = summary.loc[mask].sort_values(["mean_mse", "std_mse"], na_position="last")
        if subset.empty:
            continue
        row = subset.iloc[0].copy()
        row["selection"] = label
        rows.append(row)
    if not rows:
        return pd.DataFrame()
    cols = [
        "selection", "param_id", "mean_mse", "std_mse", "n_folds",
        "ap_form", "ls_xy", "ls_z", "os_xyz", "ls_t", "os_t", "ls_ap", "os_ap",
        "mean_fit_elapsed", "mean_predict_elapsed",
    ]
    return pd.DataFrame(rows)[cols]

family_bests = best_rows_by_family(active_summary)
family_bests


In [ ]:

if FINAL_GP_PATH.exists() or FINAL_KNN_PATH.exists():
    final_rows = []
    for path in [FINAL_GP_PATH, FINAL_KNN_PATH]:
        if path.exists():
            final_rows.append(pd.read_csv(path))
    final_results = pd.concat(final_rows, ignore_index=True)
else:
    final_results = pd.DataFrame()
final_results


In [ ]:

if not active_summary.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    active_summary.sort_values("mean_mse")["mean_mse"].reset_index(drop=True).plot(ax=ax)
    ax.set_title(f"{active_stage} GP configs by mean geo-CV MSE")
    ax.set_xlabel("rank")
    ax.set_ylabel("mean MSE")
    plt.tight_layout()
